# 02_download_road_network_data

Check Geofabrik PBF files and batch-generate drive/walk road network files for 86 cities.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path('/Volumes/ZHITAI2T/202606osm')


In [ ]:
# Read city_boundaries, city_master, and the Geofabrik extract inventory to confirm Step 02 inputs are ready.
city_table = pd.read_csv(ROOT / 'data/01_city_boundaries_and_sample_inventory/city_sample/city_master.csv')
extracts = pd.read_csv(ROOT / 'data/02_download_road_network_data/download_inventory/country_region_download_inventory.csv')
print('cities:', city_table.shape)
print('extracts:', extracts.shape)
display(city_table[['city_id', 'city_name_en', 'iso3', 'geofabrik_extract_id']].head())
display(extracts.head())


In [ ]:
# Download status is maintained by the background manager; use this cell to check whether the 51 Geofabrik PBF files are complete.
pbf_dir = ROOT / 'data/02_download_road_network_data/OSM_snapshots'
downloaded = sorted(pbf_dir.glob('*.osm.pbf'))
print('completed pbf files:', len(downloaded))
display(pd.DataFrame({'file': [p.name for p in downloaded], 'bytes': [p.stat().st_size for p in downloaded]}).head())


In [ ]:
# Generate the task table for 86 cities x drive/walk road network builds. The manager scans PBF files by Geofabrik extract group to avoid repeated reads of large files.
import subprocess, sys
manager = ROOT / 'code_upload/02_download_road_network_data/road_network_build_manager.py'
print(subprocess.check_output([sys.executable, str(manager), '--prepare-only'], cwd=ROOT).decode().strip())
tasks = pd.read_csv(ROOT / 'data/02_download_road_network_data/road_network_build_task_table.csv')
display(tasks['status'].value_counts())
display(tasks[['task_id', 'city_name_en', 'network_type', 'geofabrik_extract_id', 'output_graphml', 'output_gpkg']].head())


In [ ]:
# Start the background road network build. Default OSM_ROAD_BUILD_MAX_WORKERS=1 reduces memory and disk pressure from simultaneous reads/writes of multiple large PBF files.
import subprocess, sys
launcher = ROOT / 'code_upload/02_download_road_network_data/launch_road_network_build.py'
print('road build pid:', subprocess.check_output([sys.executable, str(launcher)], cwd=ROOT).decode().strip())


In [ ]:
# Run this cell for later status checks; no real-time monitoring is needed.
status_path = ROOT / 'data/02_download_road_network_data/road_network_build_status_table.csv'
if status_path.exists():
    status = pd.read_csv(status_path)
    display(status['status'].value_counts())
    display(status.tail(20))
else:
    print('Road network build status table has not been generated yet.')
